# Домашняя работа №1

**ФИО**: Дергалов Никита Олегович

**Группа**: ИУ6-55Б

**Вариант**: №2

## Задание:

- Рассчитайте средний рейтинг товаров из набора данных.
- Сопоставьте полученные данные из предыдущего пункта с наименованием товаров.
- Сформируйте RDD товаров с рейтингом меньшим 3. Выведите топ-10 товаров с наименьшим рейтингом.
- Сохраните результат в постоянное хранилище.
Замечание: для парсинга товаров используйте функцию eval.

### Подключаем Spark

In [ ]:
from pyspark import SparkContext

In [3]:
sc = SparkContext(appName="AmazonAverageRating").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/21 20:15:18 WARN Utils: Your hostname, Kvasik, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface enp7s0)
25/11/21 20:15:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/21 20:15:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/21 20:15:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Рассчитываем средний рейтинг товаров из набора данных.

In [4]:
reviews_path = "Electronics_5.json"

def parse_review(line):
    try:
        return eval(line)
    except:
        return None
    
reviews_rdd = sc.textFile(reviews_path).map(parse_review).filter(lambda x: x is not None)
reviews_rdd.first()

{'reviewerID': 'AO94DHGC771SJ',
 'asin': '0528881469',
 'reviewerName': 'amazdnu',
 'helpful': [0, 0],
 'reviewText': 'We got this GPS for my husband who is an (OTR) over the road trucker.  Very Impressed with the shipping time, it arrived a few days earlier than expected...  within a week of use however it started freezing up... could of just been a glitch in that unit.  Worked great when it worked!  Will work great for the normal person as well but does have the "trucker" option. (the big truck routes - tells you when a scale is coming up ect...)  Love the bigger screen, the ease of use, the ease of putting addresses into memory.  Nothing really bad to say about the unit with the exception of it freezing which is probably one in a million and that\'s just my luck.  I contacted the seller and within minutes of my email I received a email back with instructions for an exchange! VERY impressed all the way around!',
 'overall': 5.0,
 'summary': 'Gotta have GPS!',
 'unixReviewTime': 13701

In [5]:
# Для среднего рейтинга по каждому товару получаем id и рейтинг товара: (asin, (rating, 1))
product_ratings = reviews_rdd.map(lambda r: (r["asin"], (r["overall"], 1)))
product_ratings.first()

('0528881469', (5.0, 1))

In [6]:
# Суммируем рейтинги и количество отзывов для каждого товара
rating_sum_count = product_ratings.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
rating_sum_count.first()

('9985511476', (88.0, 19))

In [7]:
# Получаем средний рейтинг каждого товара
average_ratings = rating_sum_count.mapValues(lambda x: x[0] / x[1])
average_ratings.first()

('9985511476', 4.631578947368421)

### Сопоставляем полученные данные из предыдущего пункта с наименованием товаров

In [8]:
metadata_path = "meta_Electronics.json"

def parse_meta(line):
    try:
        return eval(line)
    except:
        return None

meta_rdd = sc.textFile(metadata_path).map(parse_meta).filter(lambda x: x is not None)
# (asin, title)
product_titles = meta_rdd.map(lambda m: (m["asin"], m.get("title", "")))

# Делаем join по asin: (asin, (average_rating, title))
ratings_with_titles = average_ratings.join(product_titles)

In [9]:
ratings_with_titles.lookup('9985511476')

[(4.631578947368421,
  'Professional Kingston MicroSDHC 4GB Card for Garmin nuvi 1450')]

### Сформируем RDD товаров с рейтингом меньшим 3 и выведим топ-10 товаров с наименьшим рейтингом.

In [10]:
# Фильтруем товары с рейтингом < 3
low_rating_products = ratings_with_titles.filter(lambda x: x[1][0] < 3)
# Сортируем по возрастанию рейтинга и берем топ-10
top10_worst = low_rating_products.takeOrdered(10, key=lambda x: x[1][0])

for asin, (avg_rating, title) in top10_worst:
    print(f"ASIN: {asin}, Rating: {avg_rating}, Title: {title}")

ASIN: B00111JODG, Rating: 1.0, Title: StarTech HDMISPL1HH 1 feet Standard HDMI Cable - 1x HDMI (M) to 2x HDMI (F) (Discontinued by Manufacturer)
ASIN: B000H13L4Y, Rating: 1.0, Title: ATI TV Wonder 200 PCI Video Card w/PVR Capabilities
ASIN: B000NNFS4C, Rating: 1.0, Title: RCA DRC8335 DVD Recorder &amp; VCR Combo With Built-In Tuner
ASIN: B000F1ORW6, Rating: 1.0, Title: GE 24746 Futura HDTV Ready Antenna
ASIN: B003KIQTXG, Rating: 1.0, Title: NEEWER&reg; Photographic Barn Door &amp; Honeycomb Grid &amp; Gel Set for Alienbees Alienbee Flash
ASIN: B004GGRPGG, Rating: 1.0, Title: Rapid USB Charger Adapters include: Wall and Car + USB Cable for Samsung Galaxy TAB (P1000)
ASIN: B00000JBIA, Rating: 1.0, Title: Agfa ePhoto SMILE 0.2MP Digital Camera
ASIN: B0013WI5SS, Rating: 1.0, Title: Dynex-DX-AP100 Adapter Mini DVI to Mini-DIN
ASIN: B001T9N0R6, Rating: 1.0, Title: Zeikos 57-in-1 USB 2.0 Flash Memory Card Reader ZE-CR201
ASIN: B004K9CL6S, Rating: 1.0, Title: NEW SYLVANIA HD1Z SDSDHCMMC 720P H

In [12]:
output_path = "output/amazon_low_rating_products"
# Сохраняем как текстовый файл — по строке на товар
low_rating_products.saveAsTextFile(output_path)